<div style = "
    background-color: #242120;
    border: 2px solid black;
    box-shadow: 2px 2px 2px 2px black;
    text-align: center;
">

<h1>Homework</h1>

</div>

## Table of Content

- [Question 1]()
- [Question 2]()
- [Question 3]()
- [Question 4]()
- [Question 5]()
- [Question 6]()

<hr>

In [7]:
import json
from kafka import KafkaProducer, KafkaConsumer
import dataclasses
import time
import pandas as pd
import sys
sys.path.insert(0, '../notebooks')

In [9]:
from dataclasses import dataclass

# Green Ride dataclass for homework
@dataclass
class Green_Ride:
    lpep_pickup_datetime: str
    lpep_dropoff_datetime: str
    PULocationID: int
    DOLocationID: int
    passenger_count: int
    trip_distance: float
    tip_amount: float
    total_amount: float

def ride_from_row_green(row):
    return Green_Ride(
        lpep_pickup_datetime=row['lpep_pickup_datetime'].isoformat(),
        lpep_dropoff_datetime=row['lpep_dropoff_datetime'].isoformat(),
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        passenger_count=int(row['passenger_count']) if pd.notna(row['passenger_count']) else 0,
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount'])
    )

def ride_serializer_green(ride):
    """Serializes any dataclass object to JSON"""
    ride_dict = dataclasses.asdict(ride)
    ride_json = json.dumps(ride_dict).encode('utf-8')
    return ride_json

def ride_deserialzer_green(ride_json):
    """Deserializes JSON back to a Green_Ride dataclass object"""
    ride_dict = json.loads(ride_json.decode('utf-8'))
    return Green_Ride(**ride_dict)

In this homework we will use redpanda, flink and postgresql to stream data from the green_tripdata_2025_10.parquet file.
The infrastructure set up in the course is used

## Question 1

Run <span style = "background-color: #C3C41F; color: black; padding: 10px">rpk version</span> inside the Redpanda container:

In [6]:
!docker exec -it pyflink_streaming_workshop-redpanda-1 rpk version

rpk version: v25.3.9
Git ref:     836b4a36ef6d5121edbb1e68f0f673c2a8a244e2
Build date:  2026 Feb 26 07 48 21 Thu
OS/Arch:     linux/amd64
Go version:  go1.24.3

Redpanda Cluster
  node-1  v25.3.9 - 836b4a36ef6d5121edbb1e68f0f673c2a8a244e2


As one can see from the output, the version of my rpk is <span style = "background-color: #C3C41F; color: black; padding: 10px">v25.3.9</span>

## Question 2

Create a topic called green trips:

In [ ]:
!docker exec -it pyflink_streaming_workshop-redpanda-1 rpk topic create green-trips

In [4]:
# Load the green tripdata parquet file
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet')

# Select only the required columns
required_cols = [
    'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID',
    'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount'
]
df = df[required_cols].copy()

# Drop rows where required non-null fields are missing
# Keep passenger_count nullable and fill it with 0 to avoid int(NaN) errors.
df = df.dropna(subset=[
    'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID',
    'DOLocationID', 'trip_distance', 'tip_amount', 'total_amount'
])
df['passenger_count'] = df['passenger_count'].fillna(0)

print(f"Loaded {len(df)} cleaned rows")

Loaded 49416 cleaned rows


In [14]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [10]:
# Create a Kafka producer
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=ride_serializer_green
)

In [11]:
# Send green taxi data to the green-trips topic
topic_name = 'green-trips'

t0 = time.time()

for idx, (_, row) in enumerate(df.iterrows(), start=1):
    ride = ride_from_row_green(row)
    producer.send(topic_name, value=ride)

    if idx % 10000 == 0:
        print(f"Queued {idx} records...")

producer.flush()

t1 = time.time()
print(f'Sent {len(df)} records in {(t1 - t0):.2f} seconds')

Queued 10000 records...
Queued 20000 records...
Queued 30000 records...
Queued 40000 records...
Sent 49416 records in 9.63 seconds


As one can see the entire process lasted about 10 seconds

## Question 3

Write a Kafka consumer that reads all messages from the green-trips topic (set auto_offset_reset='earliest').

Count how many trips have a trip_distance greater than 5.0 kilometers.

How many trips have trip_distance > 5

In [12]:
# Question 3: create a consumer that starts from the beginning of the topic
consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    group_id=None,
    value_deserializer=ride_deserialzer_green,
    consumer_timeout_ms=10000  # stop after 10s without new messages
)

In [13]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [14]:
# Create the table (idempotent - safe to re-run)
cur.execute("""
    CREATE TABLE IF NOT EXISTS green_trips (
        id                     SERIAL PRIMARY KEY,
        lpep_pickup_datetime   TIMESTAMP,
        lpep_dropoff_datetime  TIMESTAMP,
        pu_location_id         INT,
        do_location_id         INT,
        passenger_count        INT,
        trip_distance          FLOAT,
        tip_amount             FLOAT,
        total_amount           FLOAT
    )
""")
print("Table green_trips ready.")

Table green_trips ready.


In [15]:
# Consume all messages from green-trips and insert into PostgreSQL
from datetime import datetime, timezone

def to_dt(val):
    """Accept either an ISO string or epoch-milliseconds int."""
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return datetime.fromtimestamp(val / 1000, tz=timezone.utc).replace(tzinfo=None)
    return val  # already an ISO string; psycopg2 parses it automatically

for message in consumer:
    ride = message.value  # dict from JSON deserializer

    cur.execute(
        """INSERT INTO green_trips
           (lpep_pickup_datetime, lpep_dropoff_datetime,
            pu_location_id, do_location_id, passenger_count,
            trip_distance, tip_amount, total_amount)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (
            to_dt(ride.lpep_pickup_datetime),
            to_dt(ride.lpep_dropoff_datetime),
            ride.PULocationID,
            ride.DOLocationID,
            ride.passenger_count,
            ride.trip_distance,
            ride.tip_amount,
            ride.total_amount,
        )
    )

consumer.close()
print("Done.")

Done.


In [16]:
# Query: total rows and count of trips with distance > 5
cur.execute("""
    SELECT
        COUNT(*)                                        AS total_trips,
        SUM(CASE WHEN trip_distance > 5.0 THEN 1 END)  AS trips_gt_5
    FROM green_trips
""")
row = cur.fetchone()
print(f"Total trips in DB:            {row[0]}")
print(f"Trips with distance > 5.0 km: {row[1]}")

# Preview the first few rows
cur.execute("SELECT * FROM green_trips LIMIT 5")
pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

Total trips in DB:            49416
Trips with distance > 5.0 km: 8506


,id,lpep_pickup_datetime,lpep_dropoff_datetime,pu_location_id,do_location_id,passenger_count,trip_distance,tip_amount,total_amount
0,1,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1,0.70,1.70,10.00
1,2,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1,1.61,2.78,16.68
2,3,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1,0.00,2.20,13.20
3,4,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1,10.37,11.31,67.85
4,5,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1,4.07,6.82,34.12


## Question 4 - 6


The job in question 4 yielded a result of 74 for the PULOcation ID

Sadly I didn't manage to run jobs 5-6 due to some complications.